In [ ]:
%reload_ext autoreload
%autoreload 2


import functools
print = functools.partial(print, flush=True)

import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm


import flexiznam as flz
from cottage_analysis.analysis import common_utils, size_control, find_depth_neurons
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting import size_control_plots, depth_selectivity_plots
from cottage_analysis.summary_analysis import summary_utils

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

In [ ]:
project = "hey2_3d-vision_foodres_20220101"
flexilims_session = flz.get_flexilims_session(project)
from v1_depth_map.paths import get_figures_roots

READ_ROOT, SAVE_ROOT = get_figures_roots(
    flexilims_session, fig_subdir="fig_size_control"
)
os.makedirs(SAVE_ROOT, exist_ok=True)

In [ ]:
# Load data
project = "hey2_3d-vision_foodres_20220101"

# Load example session
session_name = "PZAH10.2d_S20230822"
flexilims_session = flz.get_flexilims_session(project)

vs_df_example, trials_df_example = size_control.sync_all_recordings(
    session_name=session_name,
    flexilims_session=flexilims_session,
    project=project,
    filter_datasets={"anatomical_only": 3, "ast_neuropil": False},
    recording_type="two_photon",
    protocol_base="SizeControl",
    photodiode_protocol=5,
    return_volumes=True,
)

neurons_ds_example = pipeline_utils.create_neurons_ds(
    session_name=session_name,
    flexilims_session=flexilims_session,
    project=None,
    conflicts="skip",
)
neurons_df_example = pd.read_pickle(
    neurons_ds_example.path_full.parent / "neurons_df.pickle"
)

# Load neurons_df of all sessions
session_list = ["PZAH10.2d_S20230822", "PZAH10.2f_S20230815", "PZAH10.2f_S20230907"]
flexilims_session = flz.get_flexilims_session(project)
neurons_df_all = summary_utils.concatenate_all_neurons_df(
    flexilims_session=flexilims_session,
    session_list=session_list,
    filename="neurons_df.pickle",
    cols=None,
    read_iscell=True,
    verbose=True,
)

In [ ]:
common_utils.add_one_sided_spearman_significance(
    neurons_df_all,
    rval_col="depth_tuning_test_spearmanr_rval_closedloop",
    pval_col="depth_tuning_test_spearmanr_pval_closedloop",
    out_col="is_depth_neuron",
)
select_neurons = (neurons_df_all.iscell == 1) & (neurons_df_all.is_depth_neuron)
print(f"Number of neurons: {select_neurons.sum()}")
ROIS = [453, 133, 448]
# compute ylims for neurons' depth tuning
ylims_all = np.zeros((len(ROIS), 2, 4, 2))
for iroi, roi in enumerate(ROIS):
    for iparam, param in enumerate(["depth", "size"]):
        mean_dff_arr = find_depth_neurons.average_dff_for_all_trials(
            trials_df=trials_df_example,
            rs_thr=None,
            rs_thr_max=None,
            still_only=False,
            still_time=0,
            frame_rate=15,
            closed_loop=1,
            param="size",
        )[:, :, roi]
        CI_low, CI_high = common_utils.get_bootstrap_ci(mean_dff_arr)
        ylim = [round(np.nanmin(CI_low), 1), round(np.nanmax(CI_high), 1)]
        ylims_all[iroi, iparam, 0, :] = ylim

        for isize, size in enumerate([5, 10, 20]):
            mean_dff_arr = find_depth_neurons.average_dff_for_all_trials(
                trials_df=trials_df_example[trials_df_example["size"] == size],
                rs_thr=None,
                rs_thr_max=None,
                still_only=False,
                still_time=0,
                frame_rate=15,
                closed_loop=1,
                param="size",
            )[:, :, roi]
            CI_low, CI_high = common_utils.get_bootstrap_ci(mean_dff_arr)
            ylim = [np.nanmin(CI_low), np.nanmax(CI_high)]
            ylims_all[iroi, iparam, isize + 1, :] = ylim

In [ ]:
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT
from cottage_analysis.plotting import style

fig = plt.figure(figsize=(18 * CM, 18 * CM))

# Add panel letters
fig.text(0.01, 0.88, "A", fontsize=FONTSIZE_DICT["panel"], fontweight="bold")
fig.text(0.01, 0.63, "B", fontsize=FONTSIZE_DICT["panel"], fontweight="bold")
fig.text(0.01, 0.35, "C", fontsize=FONTSIZE_DICT["panel"], fontweight="bold")

# Example cells: depth tuning at different sphere sizes
ROIS = [453, 133, 448]
ylims_all[0, 0, :, 0] = 0.2
ylims_all[1, 0, :, 0] = 0.0
ylims_all[2, 0, :, 0] = 0.0

for iroi, roi in enumerate(ROIS):
    for iparam, param in enumerate(["depth"]):
        ax = fig.add_axes([0.08 + iroi * 0.31, 0.73, 0.18, 0.15])

        depth_tuning_kwargs = dict(
            rs_thr=None,
            plot_fit=True,
            plot_smooth=False,
            linewidth=1.5,
            closed_loop=1,
            fontsize_dict=FONTSIZE_DICT,
            markersize=10,
            markeredgecolor="w",
            clip_on=False,
            linecolor="k",
        )

        depth_selectivity_plots.plot_depth_tuning_curve(
            neurons_df=neurons_df_example,
            trials_df=trials_df_example,
            roi=roi,
            param=param,
            use_col=f"{param}_tuning_popt_closedloop",
            min_sigma=0.5,
            ylim=[
                np.nanmin(ylims_all[iroi, 0, :, 0]),
                common_utils.ceil(np.nanmax(ylims_all[iroi, 0, :, 1]), 1),
            ],
            **depth_tuning_kwargs,
        )
        if iroi != 1:
            ax.set_xlabel("")
        if iroi == 1:
            ax.set_yticks([0.0, 1.5])
        elif iroi == 2:
            ax.set_yticks([0, 1])

        ax = fig.add_axes([0.08 + iroi * 0.31, 0.48, 0.18, 0.15])
        depth_tuning_kwargs = dict(
            rs_thr=None,
            plot_fit=True,
            plot_smooth=False,
            linewidth=1.5,
            closed_loop=1,
            fontsize_dict=FONTSIZE_DICT,
            markersize=10,
            markeredgecolor="w",
            clip_on=False,
        )
        for size, linecolor in zip([5, 10, 20], ["skyblue", "royalblue", "navy"]):
            depth_selectivity_plots.plot_depth_tuning_curve(
                neurons_df=neurons_df_example,
                trials_df=trials_df_example[trials_df_example["size"] == size],
                roi=roi,
                param=param,
                use_col=f"{param}_tuning_popt_size{size}",
                min_sigma=0.5,
                ylim=[
                    np.nanmin(ylims_all[iroi, 0, :, 0]),
                    common_utils.ceil(np.nanmax(ylims_all[iroi, 0, :, 1]), 1),
                ],
                label=f"{size} degrees",
                linecolor=linecolor,
                **depth_tuning_kwargs,
            )
        if iroi == 0:
            plt.legend(fontsize=FONTSIZE_DICT["legend"], frameon=False, handlelength=1)
        if iroi != 1:
            ax.set_xlabel("")
        if iroi == 1:
            ax.set_yticks([0.0, 1.5])
        elif iroi == 2:
            ax.set_yticks([0, 1])

# Scatter plot of preferred depths at different visual angles
common_utils.add_one_sided_spearman_significance(
    neurons_df_all,
    rval_col="depth_tuning_test_spearmanr_rval_closedloop",
    pval_col="depth_tuning_test_spearmanr_pval_closedloop",
    out_col="is_depth_neuron",
)
select_neurons = (neurons_df_all.iscell == 1) & (neurons_df_all.is_depth_neuron)
size_control_plots.plot_preferred_depths_sizes_scatter(
    neurons_df=neurons_df_all[select_neurons],
    sizes=[5, 10, 20],
    plot_x=0.08,
    plot_y=0.08,
    plot_width=0.31,
    plot_height=0.31,
    fontsize_dict=FONTSIZE_DICT,
    scatter_kwargs=dict(s=10, c="k", alpha=0.4, edgecolors="none"),
)

# style.savefig keeps SVG text at the right size in Illustrator - see
# cottage_analysis.plotting.style.expand_font_shorthand for why.
style.savefig(
    SAVE_ROOT / "fig_size_tuning.svg",
    bbox_inches="tight",
    dpi=300,
    verbose=True,
    fig=fig,
)
print(f"Figure saved to {SAVE_ROOT/'fig_size_tuning.svg'}")

## stat

In [ ]:
from scipy.stats import spearmanr

# Format and print stats matching the manuscript text
if "mouse" not in neurons_df_all.columns:
    neurons_df_all["mouse"] = neurons_df_all["session"].apply(lambda x: x.split("_")[0])

df = neurons_df_all[select_neurons]
n_boots = 20000
xcol = ["preferred_depth_size5", "preferred_depth_size5", "preferred_depth_size10"]
ycol = ["preferred_depth_size10", "preferred_depth_size20", "preferred_depth_size20"]
labels = [
    "5 degree vs. 10 degree spheres",
    "5 degree vs. 20 degree spheres",
    "10 vs. 20 degree spheres",
]

np.random.seed(0)
r_ratio, dist_ratio = common_utils.hierarchical_bootstrap_stats(
    df,
    n_boots,
    xcol=xcol,
    ycol=ycol,
    resample_cols=["mouse", "session"],
    correlation=False,
    difference=False,
    ratio=True,
)

stats_strings = []
for i in range(3):
    pval_ratio = common_utils.calculate_pval_from_bootstrap(dist_ratio[:, i], value=1)
    sp = spearmanr(df[xcol[i]], df[ycol[i]])
    if sp.pvalue < 0.0001:
        p_corr_str = "p_{correlation} < 0.0001"
    else:
        p_corr_str = f"p_{{correlation}} = {sp.pvalue:.4f}"

    s = f"{labels[i]}, median ratio = {r_ratio[i]:.2f}, $p_{{ratio}} = {pval_ratio:.3f}$, $r = {sp.statistic:.3f}$, ${p_corr_str}$"
    stats_strings.append(s)

print("Preferred virtual depth mapped with:")
print("\n".join(stats_strings))